# Coffee Shop Chain Operations & Customer Feedback

This notebook builds a fictional UK coffee shop chain SQLite database for a Data Mining and Discovery assignment.

It creates seven related tables, includes nominal, ordinal, interval and ratio variables, uses foreign keys and compound keys, and generates more than 1,000 transaction records. The dataset is synthetic and suitable for database and data-mining practice.

In [3]:
# -----------------------------------------------------------
# Coffee Shop Chain Operations & Customer Feedback - SQLite build
# -----------------------------------------------------------

!pip install faker -q

import sqlite3, random, os
from faker import Faker
from datetime import timedelta, date, datetime

fake = Faker('en_GB')
Faker.seed(42)
random.seed(42)

DB = 'coffee_shop.db'
if os.path.exists(DB):
    os.remove(DB)

conn = sqlite3.connect(DB)
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()


# -----------------------------------------------------------
# schema
# six tables:
#   Suppliers      - coffee/food suppliers
#   Branches       - coffee shop locations
#   Products       - menu items (composite PK)
#   Customers      - loyalty/customer dimension
#   Orders         - transaction header
#   Order_Items    - transaction lines (composite PK)
#   Feedback       - customer review data
#
# stat type tags per column: nominal / ordinal / interval / ratio
# -----------------------------------------------------------

cur.executescript("""
CREATE TABLE Suppliers (
    supplier_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    supplier_name     TEXT NOT NULL,
    city              TEXT NOT NULL,                  -- nominal
    supplier_type     TEXT NOT NULL CHECK (supplier_type IN ('coffee roaster','bakery','dairy','packaging')),
    service_tier      TEXT NOT NULL CHECK (service_tier IN ('preferred','standard','backup')), -- ordinal
    year_registered   INTEGER CHECK (year_registered BETWEEN 1980 AND 2026) -- interval
);

CREATE TABLE Branches (
    branch_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    branch_name       TEXT NOT NULL UNIQUE,           -- nominal
    city              TEXT NOT NULL,                  -- nominal
    postcode_area     TEXT NOT NULL,
    seating_capacity  INTEGER NOT NULL CHECK (seating_capacity >= 0), -- ratio
    has_drive_through INTEGER NOT NULL CHECK (has_drive_through IN (0,1))
);

CREATE TABLE Products (
    product_code      TEXT NOT NULL,
    size              TEXT NOT NULL CHECK (size IN ('Small','Regular','Large','Single','Slice','Bottle','Add-on')),
    product_name      TEXT NOT NULL,                  -- nominal
    category          TEXT NOT NULL CHECK (category IN ('Coffee','Tea','Cold Drinks','Bakery','Food','Extras')),
    dietary_type      TEXT NOT NULL CHECK (dietary_type IN ('Regular','Vegetarian','Vegan')),
    unit_price_gbp    REAL NOT NULL CHECK (unit_price_gbp > 0), -- ratio
    caffeine_mg       INTEGER NOT NULL CHECK (caffeine_mg >= 0), -- ratio
    supplier_id       INTEGER,
    PRIMARY KEY (product_code, size),                 -- COMPOSITE PK #1
    FOREIGN KEY (supplier_id) REFERENCES Suppliers(supplier_id)
);

CREATE TABLE Customers (
    customer_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_ref      TEXT UNIQUE NOT NULL,           -- nominal identifier
    full_name         TEXT NOT NULL,
    date_of_birth     DATE NOT NULL,                  -- interval date
    home_city         TEXT,                           -- modified to allow NULLs for deliberate missingness exercise
    loyalty_tier      TEXT NOT NULL CHECK (loyalty_tier IN ('Bronze','Silver','Gold','Platinum')), -- ordinal
    signup_date       DATE NOT NULL,
    preferred_channel TEXT NOT NULL CHECK (preferred_channel IN ('In-store','Mobile App','Click & Collect','Delivery'))
);

CREATE TABLE Orders (
    order_ref         TEXT NOT NULL,
    order_line_group  INTEGER NOT NULL CHECK (order_line_group > 0),
    customer_id       INTEGER NOT NULL,
    branch_id         INTEGER NOT NULL,
    order_datetime    TEXT NOT NULL,
    payment_method    TEXT NOT NULL CHECK (payment_method IN ('Cash','Debit Card','Credit Card','Apple Pay','Google Pay')), -- nominal
    order_channel     TEXT NOT NULL CHECK (order_channel IN ('In-store','Mobile App','Click & Collect','Delivery')),
    temperature_c    REAL NOT NULL,                   -- interval (Celsius)
    discount_percent  REAL NOT NULL CHECK (discount_percent BETWEEN 0 AND 50), -- ratio/percentage
    order_total_gbp   REAL CHECK (order_total_gbp >= 0), -- ratio
    satisfaction_band TEXT NOT NULL CHECK (satisfaction_band IN ('Very Poor','Poor','Average','Good','Excellent')), -- ordinal
    PRIMARY KEY (order_ref, order_line_group),         -- COMPOSITE PK #2
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
    FOREIGN KEY (branch_id) REFERENCES Branches(branch_id)
);

CREATE TABLE Order_Items (
    order_ref         TEXT NOT NULL,
    order_line_group  INTEGER NOT NULL,
    product_code      TEXT NOT NULL,
    size              TEXT NOT NULL,
    quantity          INTEGER NOT NULL CHECK (quantity > 0), -- ratio
    unit_price_gbp    REAL NOT NULL CHECK (unit_price_gbp > 0),
    line_total_gbp    REAL NOT NULL CHECK (line_total_gbp >= 0),
    PRIMARY KEY (order_ref, order_line_group, product_code, size), -- COMPOSITE PK #3
    FOREIGN KEY (order_ref, order_line_group) REFERENCES Orders(order_ref, order_line_group),
    FOREIGN KEY (product_code, size) REFERENCES Products(product_code, size)
);

CREATE TABLE Feedback (
    feedback_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    order_ref         TEXT NOT NULL,
    order_line_group  INTEGER NOT NULL,
    feedback_date     DATE NOT NULL,
    customer_rating   INTEGER NOT NULL CHECK (customer_rating BETWEEN 1 AND 5), -- ordinal
    service_speed     TEXT NOT NULL CHECK (service_speed IN ('Very Slow','Slow','Average','Fast','Very Fast')), -- ordinal
    review_channel    TEXT NOT NULL CHECK (review_channel IN ('App','Email Survey','Receipt QR','Website')),
    review_text       TEXT NOT NULL,
    would_recommend   INTEGER NOT NULL CHECK (would_recommend IN (0,1)),
    FOREIGN KEY (order_ref, order_line_group) REFERENCES Orders(order_ref, order_line_group)
);
""")
conn.commit()


# -----------------------------------------------------------
# generate data (synthetic UK coffee chain data)
# -----------------------------------------------------------

supplier_names = [
    ("North Star Coffee Roasters","Leeds","coffee roaster"),
    ("Union Hand-Roasted Coffee","London","coffee roaster"),
    ("Origin Coffee","Bristol","coffee roaster"),
    ("Riverbank Bakery Supply","Reading","bakery"),
    ("Dairy Crest Wholesale","Manchester","dairy"),
    ("Green Fields Plant Milk","Oxford","dairy"),
    ("PackRight UK","Birmingham","packaging"),
    ("FreshBake Partners","Brighton","bakery"),
    ("Ethical Bean Trading","London","coffee roaster"),
    ("City Foodservice","Liverpool","bakery"),
    ("Cotswold Dairy Supply","Gloucester","dairy"),
    ("EcoServe Packaging","Nottingham","packaging")
]
for name, city, stype in supplier_names:
    cur.execute("""INSERT INTO Suppliers
        (supplier_name, city, supplier_type, service_tier, year_registered)
        VALUES (?,?,?,?,?)""",
        (name, city, stype, random.choices(['preferred','standard','backup'],[3,5,2])[0],
         random.randint(1985,2024)))

branch_data = [
    ("Oxford Street","London","W1",35,1),("Camden High Street","London","NW1",28,0),
    ("Shoreditch","London","E1",30,0),("King's Cross","London","N1",42,0),
    ("Bristol Harbourside","Bristol","BS1",32,0),("Manchester Piccadilly","Manchester","M1",40,0),
    ("Leeds City Centre","Leeds","LS1",34,0),("Birmingham New Street","Birmingham","B2",38,0),
    ("Liverpool One","Liverpool","L1",36,0),("Nottingham Market Square","Nottingham","NG1",26,0),
    ("Brighton Seafront","Brighton","BN1",30,0),("Reading Station","Reading","RG1",24,0),
    ("Oxford High Street","Oxford","OX1",27,0),("Guildford High Street","Guildford","GU1",22,0),
    ("Thames Ditton Village","Thames Ditton","KT7",20,1)
]
cur.executemany("""INSERT INTO Branches
    (branch_name, city, postcode_area, seating_capacity, has_drive_through)
    VALUES (?,?,?,?,?)""", branch_data)

# product menu - each product code + size is unique
menu = [
    ("ESP","Small","Espresso","Coffee","Regular",2.20,64),
    ("AME","Regular","Americano","Coffee","Vegan",2.80,150),
    ("FW","Regular","Flat White","Coffee","Regular",3.45,130),
    ("CAP","Regular","Cappuccino","Coffee","Regular",3.40,120),
    ("LAT","Regular","Latte","Coffee","Regular",3.55,150),
    ("LAT","Large","Latte","Coffee","Regular",4.05,190),
    ("MOC","Regular","Mocha","Coffee","Regular",3.85,175),
    ("ICL","Regular","Iced Latte","Cold Drinks","Regular",3.75,140),
    ("CB","Regular","Cold Brew","Cold Drinks","Vegan",3.60,180),
    ("MAT","Regular","Matcha Latte","Tea","Regular",3.90,70),
    ("TEA","Regular","English Breakfast Tea","Tea","Vegan",2.50,45),
    ("CHAI","Regular","Chai Latte","Tea","Regular",3.50,70),
    ("HC","Regular","Hot Chocolate","Coffee","Regular",3.30,10),
    ("MUFF","Single","Blueberry Muffin","Bakery","Vegetarian",2.75,0),
    ("BROWN","Single","Chocolate Brownie","Bakery","Vegetarian",2.95,0),
    ("CROIS","Single","Butter Croissant","Bakery","Vegetarian",2.60,0),
    ("BAN","Slice","Banana Bread","Bakery","Vegan",3.00,0),
    ("COOKIE","Single","Vegan Cookie","Bakery","Vegan",2.50,0),
    ("TOAST","Single","Ham & Cheese Toastie","Food","Regular",4.95,0),
    ("PANINI","Single","Mozzarella Panini","Food","Vegetarian",5.25,0),
    ("SALAD","Single","Chicken Salad","Food","Regular",6.20,0),
    ("AVO","Single","Avocado Toast","Food","Vegan",5.75,0),
    ("WATER","Bottle","Sparkling Water","Cold Drinks","Vegan",1.80,0),
    ("JUICE","Bottle","Orange Juice","Cold Drinks","Vegan",2.60,0),
    ("OAT","Add-on","Oat Milk Upgrade","Extras","Vegan",0.50,0),
    ("SHOT","Add-on","Extra Espresso Shot","Extras","Vegan",0.70,64)
]
for p in menu:
    code, size, name, category, dietary, price, caffeine = p
    sid = random.randint(1,len(supplier_names))
    cur.execute("""INSERT INTO Products
        (product_code,size,product_name,category,dietary_type,unit_price_gbp,caffeine_mg,supplier_id)
        VALUES (?,?,?,?,?,?,?,?)""", (code,size,name,category,dietary,price,caffeine,sid))

# customers - 1,200 fictional customers
cities = ["London","Bristol","Manchester","Leeds","Birmingham","Liverpool",
          "Nottingham","Brighton","Reading","Oxford","Guildford","Thames Ditton"]
for i in range(1200):
    dob = fake.date_of_birth(minimum_age=18, maximum_age=85)
    signup = fake.date_between(start_date='-5y', end_date='today')
    tier = random.choices(['Bronze','Silver','Gold','Platinum'],[48,29,17,6])[0]
    pref = random.choice(['In-store','Mobile App','Click & Collect','Delivery'])
    cur.execute("""INSERT INTO Customers
        (customer_ref,full_name,date_of_birth,home_city,loyalty_tier,signup_date,preferred_channel)
        VALUES (?,?,?,?,?,?,?)""",
        (f"CUST-{i+1:05d}", fake.name(), str(dob), random.choice(cities), tier, str(signup), pref))

# orders - 3,500 main transactions across 15 branches
product_rows = cur.execute("SELECT product_code,size,unit_price_gbp FROM Products").fetchall()
orders_for_feedback = []
start = datetime(2024,1,1,7,0)
for i in range(3500):
    ref = f"BB-{i+1:07d}"
    customer_id = random.randint(1,1200)
    branch_id = random.randint(1,len(branch_data))
    dt = start + timedelta(minutes=random.randint(0, 950*24*60))
    dt = dt.replace(hour=random.choices(list(range(7,21)),[2,3,5,7,8,9,10,10,10,9,8,7,5,3])[0],
                    minute=random.choice([0,10,20,30,40,50]))
    channel = random.choices(['In-store','Mobile App','Click & Collect','Delivery'],[55,20,18,7])[0]
    payment = random.choices(['Cash','Debit Card','Credit Card','Apple Pay','Google Pay'],[10,34,18,25,13])[0]
    temp = round(random.uniform(-1,31),1)
    discount = random.choices([0,5,10,15,20],[66,12,13,6,3])[0]
    picks = random.sample(product_rows, k=random.choices([1,2,3,4],[18,44,28,10])[0])
    line_items = []
    subtotal = 0
    for code,size,price in picks:
        qty = random.choices([1,2,3],[78,19,3])[0]
        line_total = round(qty*price,2)
        subtotal += line_total
        line_items.append((code,size,qty,price,line_total))
    total = round(subtotal*(1-discount/100),2)
    sat = random.choices(['Very Poor','Poor','Average','Good','Excellent'],[3,5,17,40,35])[0]
    cur.execute("""INSERT INTO Orders VALUES (?,?,?,?,?,?,?,?,?,?,?)""",
        (ref,1,customer_id,branch_id,dt.isoformat(sep=' '),payment,channel,temp,discount,total,sat))
    for code,size,qty,price,line_total in line_items:
        cur.execute("""INSERT INTO Order_Items VALUES (?,?,?,?,?,?,?)""",
            (ref,1,code,size,qty,price,line_total))
    orders_for_feedback.append((ref,1,dt.date()))

# feedback - 1,900 reviews linked to orders
positive = ["Great coffee and friendly staff.","Quick service and a lovely atmosphere.",
            "The coffee was excellent and well made.","Fresh pastry and helpful service.",
            "Convenient location and good value.","Staff were welcoming and collection was quick."]
neutral = ["Coffee was fine, but the queue was a little long.","Good overall, although the seating area was busy.",
           "Average visit; the drink was as expected.","The order was correct but took longer than expected."]
negative = ["My drink was lukewarm when served.","The queue was slow and staff seemed rushed.",
            "The order was incorrect and had to be remade.","The tables needed cleaning.",
            "The pastry did not taste fresh."]
for ref,line_group,order_date in random.sample(orders_for_feedback,1900):
    rating = random.choices([1,2,3,4,5],[4,7,16,35,38])[0]
    text = random.choice(positive if rating >= 4 else neutral if rating == 3 else negative)
    speed = random.choices(['Very Slow','Slow','Average','Fast','Very Fast'],[3,9,28,42,18])[0]
    recommend = 1 if rating >= 4 and random.random() < .94 else 0
    cur.execute("""INSERT INTO Feedback
        (order_ref,order_line_group,feedback_date,customer_rating,service_speed,
         review_channel,review_text,would_recommend)
        VALUES (?,?,?,?,?,?,?,?)""",
        (ref,line_group,str(order_date+timedelta(days=random.randint(0,5))),
         rating,speed,random.choice(['App','Email Survey','Receipt QR','Website']),text,recommend))

conn.commit()


# -----------------------------------------------------------
# DELIBERATE MISSING VALUES - for data cleaning practice
# -----------------------------------------------------------
cur.execute("UPDATE Suppliers SET year_registered=NULL WHERE supplier_id IN (SELECT supplier_id FROM Suppliers ORDER BY RANDOM() LIMIT 1)")
cur.execute("UPDATE Customers SET home_city=NULL WHERE customer_id IN (SELECT customer_id FROM Customers ORDER BY RANDOM() LIMIT 18)")
cur.execute("UPDATE Orders SET order_total_gbp=NULL WHERE rowid IN (SELECT rowid FROM Orders ORDER BY RANDOM() LIMIT 45)")
conn.commit()


# -----------------------------------------------------------
# DELIBERATE DUPLICATE-LIKE EVENTS
# repeat customers may make a similar purchase on a later transaction.
# Add copies of a small sample with new order references, preserving PK uniqueness.
# -----------------------------------------------------------
cur.execute("""SELECT customer_id,branch_id,order_datetime,payment_method,order_channel,
                      temperature_c,discount_percent,order_total_gbp,satisfaction_band
               FROM Orders ORDER BY RANDOM() LIMIT 35""")
duplicate_sources = cur.fetchall()
duplicate_rows = []
for i,row in enumerate(duplicate_sources):
    duplicate_rows.append((f"BB-DUP-{i+1:04d}",1)+row)
cur.executemany("INSERT INTO Orders VALUES (?,?,?,?,?,?,?,?,?,?,?)", duplicate_rows)
conn.commit()


# -----------------------------------------------------------
# summary print
# -----------------------------------------------------------
print("=== SCHEMA ===")
for table in ['Suppliers','Branches','Products','Customers','Orders','Order_Items','Feedback']:
    print(f"\n-- {table} --")
    for row in cur.execute(f"PRAGMA table_info({table})"):
        cid,name,typ,notnull,default,pk = row
        tags=[]
        if notnull: tags.append('NOT NULL')
        if pk: tags.append(f'PK({pk})')
        if default is not None: tags.append(f'DEFAULT {default}')
        print(f"  {name:<20} {typ:<10} {' '.join(tags)}")

print("\n=== ROW COUNTS ===")
for table in ['Suppliers','Branches','Products','Customers','Orders','Order_Items','Feedback']:
    print(f"  {table:<16} {cur.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]}")

print("\n=== SAMPLE ROWS ===")
for table in ['Suppliers','Branches','Products','Customers','Orders','Order_Items','Feedback']:
    print(f"\n-- {table} --")
    for row in cur.execute(f"SELECT * FROM {table} LIMIT 3"):
        print(" ",row)

print("\n=== NULL COUNTS ===")
for table,column in [('Suppliers','year_registered'),('Customers','home_city'),('Orders','order_total_gbp')]:
    n=cur.execute(f"SELECT COUNT(*) FROM {table} WHERE {column} IS NULL").fetchone()[0]
    total=cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}.{column}: {n} nulls / {total} ({100*n/total:.1f}%)")

print("\n=== FOREIGN KEY CHECK ===")
issues=cur.execute("PRAGMA foreign_key_check").fetchall()
print("  Passed" if not issues else issues)

print("\n=== COMPOSITE KEYS ===")
print("  Products: (product_code, size)")
print("  Orders: (order_ref, order_line_group)")
print("  Order_Items: (order_ref, order_line_group, product_code, size)")
print("\n=== DATA TYPE EXAMPLES ===")
print("  Nominal: payment_method, category, city")
print("  Ordinal: loyalty_tier, customer_rating, satisfaction_band")
print("  Interval: temperature_c (Celsius), date_of_birth")
print("  Ratio: order_total_gbp, quantity, unit_price_gbp, seating_capacity")

conn.close()



=== SCHEMA ===

-- Suppliers --
  supplier_id          INTEGER    PK(1)
  supplier_name        TEXT       NOT NULL
  city                 TEXT       NOT NULL
  supplier_type        TEXT       NOT NULL
  service_tier         TEXT       NOT NULL
  year_registered      INTEGER    

-- Branches --
  branch_id            INTEGER    PK(1)
  branch_name          TEXT       NOT NULL
  city                 TEXT       NOT NULL
  postcode_area        TEXT       NOT NULL
  seating_capacity     INTEGER    NOT NULL
  has_drive_through    INTEGER    NOT NULL

-- Products --
  product_code         TEXT       NOT NULL PK(1)
  size                 TEXT       NOT NULL PK(2)
  product_name         TEXT       NOT NULL
  category             TEXT       NOT NULL
  dietary_type         TEXT       NOT NULL
  unit_price_gbp       REAL       NOT NULL
  caffeine_mg          INTEGER    NOT NULL
  supplier_id          INTEGER    

-- Customers --
  customer_id          INTEGER    PK(1)
  customer_ref         TEXT  